# Oakland Businesses → Business Improvement District (BID) Spatial Join

This notebook:
1. Reads `../../data/oakland_businesses_combined.csv` and converts it into a point `GeoDataFrame` using the `latitude`/`longitude` fields.
2. Reads `../../data/geo/block_groups/OAK_BIDs.geojson` as a polygon `GeoDataFrame`.
3. Performs a spatial join, keeping only business points that fall within a BID polygon.
4. Attaches the `FID` and `BID` fields from the polygon layer to the resulting point `GeoDataFrame`.
5. For a set of categorical fields (`broad_sector`, `employee_range`, `sales_range`, `own_or_lease_da`), computes the percentage makeup of each category within each BID — excluding null values for that field — and saves one standalone CSV per field, keyed by `FID`/`BID`, so each can be joined back to the polygon file later.

In [26]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

## 1. Load businesses CSV as a point GeoDataFrame

In [27]:
businesses_path = "../../data/business/combined/oakland_businesses_combined.csv"

df = pd.read_csv(businesses_path)

# Drop rows missing coordinates, since they can't be turned into points
df = df.dropna(subset=["latitude", "longitude"]).copy()

geometry = [Point(xy) for xy in zip(df["longitude"], df["latitude"])]

businesses_gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

print(f"Loaded {len(businesses_gdf)} business points")
businesses_gdf.head()

/var/folders/69/g590bg750s935v88zgfrdm9w0000gn/T/ipykernel_29143/1958453591.py:3: DtypeWarning: Columns (2,13,15,16,32,34,37,43,45,46,47,50,51,52,53) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(businesses_path)


Loaded 71178 business points


,canonical_business_id,location_id,business_name,address,address_base,unit,zip5,latitude,longitude,broad_sector,...,franchise_da,affiliated_records_da,affiliated_locations_da,square_footage_da,credit_score_da,phone_da,own_or_lease_da,firm_or_individual_da,government_office_da,geometry
0,BIZ-000001,LOC-000001,DHL Express Servicepoint,7201 Earhart Rd,7201 EARHART RD,NaN,94621,37.737719,-122.218558,"Professional, Scientific & Technical Services",...,NaN,0.0,0.0,"5,000 - 9,999",B+,(510) 394-9641,Unknown,2.0,0.0,POINT (-122.21856 37.73772)
1,BIZ-000002,LOC-000011,C Trans Inc,9201 San Leandro St,9201 SAN LEANDRO ST,NaN,94603,37.742601,-122.184907,Transportation & Warehousing,...,NaN,0.0,0.0,"2,500 - 4,999",C+,(510) 568-0332,Unknown,2.0,0.0,POINT (-122.18491 37.7426)
2,BIZ-000003,LOC-000020,Mutual Express,1700 W Grand Ave,1700 W GRAND AVE,NaN,94607,37.818379,-122.292059,Transportation & Warehousing,...,NaN,0.0,0.0,"2,500 - 4,999",C+,(510) 465-1711,Own,2.0,0.0,POINT (-122.29206 37.81838)
3,BIZ-000004,LOC-000036,Big Family Movers,3955 Whittle Ave,3955 WHITTLE AVE,NaN,94602,37.804975,-122.210312,Transportation & Warehousing,...,NaN,0.0,0.0,"2,500 - 4,999",C,(510) 839-5239,Unknown,2.0,0.0,POINT (-122.21031 37.80498)
4,BIZ-000005,LOC-000043,Quintero Trucking,2270 Poplar St,2270 POPLAR ST,NaN,94607,37.817704,-122.286517,Transportation & Warehousing,...,NaN,0.0,0.0,"1,500 - 2,499",C+,(510) 839-7104,Own,2.0,0.0,POINT (-122.28652 37.8177)


## 2. Load BID polygons

In [28]:
bids_path = "../../data/geo/block_groups/OAK_BIDs.geojson"

bids_gdf = gpd.read_file(bids_path)

print(f"Loaded {len(bids_gdf)} BID polygons")
print("Columns:", list(bids_gdf.columns))
bids_gdf.head()

Loaded 11 BID polygons
Columns: ['FID', 'BID', 'Shp__Ar', 'Shp__Ln', 'geometry']


,FID,BID,Shp__Ar,Shp__Ln,geometry
0,1,Downtown,669747.800781,4944.705216,"MULTIPOLYGON (((-122.27278 37.80664, -122.2725..."
1,2,Chinatown,987399.226562,4847.190191,"MULTIPOLYGON (((-122.27405 37.79837, -122.2735..."
2,3,Lake Merritt,923483.507812,5909.055754,"MULTIPOLYGON (((-122.26576 37.8047, -122.26569..."
3,4,Koreatown/Northgate,573652.921875,7483.464481,"MULTIPOLYGON (((-122.26708 37.81406, -122.2673..."
4,5,Montclair,73144.023438,1706.295224,"MULTIPOLYGON (((-122.20893 37.82561, -122.2090..."


## 3. Align CRS

Make sure both layers share the same coordinate reference system before joining.

In [29]:
if businesses_gdf.crs != bids_gdf.crs:
    businesses_gdf = businesses_gdf.to_crs(bids_gdf.crs)

print("Businesses CRS:", businesses_gdf.crs)
print("BIDs CRS:", bids_gdf.crs)

Businesses CRS: EPSG:4326
BIDs CRS: EPSG:4326


## 4. Spatial join: keep only businesses that fall within a BID polygon

`predicate="within"` keeps only the business points located inside a BID polygon (an inner join, so points outside every polygon are dropped). The `FID` and `BID` fields from the polygon layer are attached to each matching point.

In [30]:
# Only keep the polygon fields we need for the join (plus geometry), to avoid
# column-name collisions with the businesses table.
ID_COLS = ["FID", "BID"]

bids_join_cols = [c for c in ID_COLS + ["geometry"] if c in bids_gdf.columns]
missing = [c for c in ID_COLS if c not in bids_gdf.columns]
if missing:
    print(f"Warning: expected field(s) not found in BID polygon file: {missing}")
    print("Available columns:", list(bids_gdf.columns))

bids_subset = bids_gdf[bids_join_cols]

businesses_in_bids = gpd.sjoin(
    businesses_gdf,
    bids_subset,
    how="inner",
    predicate="within",
)

# Drop the sjoin index bookkeeping column, if present
businesses_in_bids = businesses_in_bids.drop(columns=["index_right"], errors="ignore")

print(f"{len(businesses_in_bids)} of {len(businesses_gdf)} businesses fall within a BID polygon")
businesses_in_bids.head()

15574 of 71178 businesses fall within a BID polygon


,canonical_business_id,location_id,business_name,address,address_base,unit,zip5,latitude,longitude,broad_sector,...,affiliated_locations_da,square_footage_da,credit_score_da,phone_da,own_or_lease_da,firm_or_individual_da,government_office_da,geometry,FID,BID
13,BIZ-000014,LOC-000113,1 Seafood & Chicken Restaurant,4014 Macarthur Blvd,4014 MACARTHUR BLVD,NaN,94619,37.790435,-122.197222,Retail Trade,...,0.0,"2,500 - 4,999",B+,(510) 482-1255,Own,2.0,0.0,POINT (-122.19722 37.79044),11,Laurel
18,BIZ-000019,LOC-000158,Friends Of The OPL,933 Broadway,933 BROADWAY,NaN,94607,37.801340,-122.273120,Retail Trade,...,0.0,"2,500 - 4,999",B+,(510) 444-0473,Own,2.0,0.0,POINT (-122.27312 37.80134),1,Downtown
23,BIZ-000024,LOC-000245,Taqueria 16 De Septiembre,3438 International Blvd,3438 INTERNATIONAL BLVD,NaN,94601,37.776518,-122.223321,Accommodation & Food Services,...,0.0,"2,500 - 4,999",C,(510) 479-1590,Own,2.0,0.0,POINT (-122.22332 37.77652),8,Fruitvale
26,BIZ-000027,LOC-000301,D H Works,184 13th St,184 13TH ST,NaN,94612,37.801424,-122.265196,Other / Unclassified,...,0.0,"2,500 - 4,999",U,Not Available,Unknown,2.0,0.0,POINT (-122.2652 37.80142),2,Chinatown
28,BIZ-000029,LOC-000311,Guardian Project,1922 Webster St,1922 WEBSTER ST,NaN,94612,37.807086,-122.266042,Health Care & Social Assistance,...,0.0,"2,500 - 4,999",I,Not Available,Unknown,2.0,0.0,POINT (-122.26604 37.80709),3,Lake Merritt


## 5. Inspect result

In [31]:
businesses_in_bids[ID_COLS].value_counts()

FID  BID                
1    Downtown               3237
3    Lake Merritt           2540
7    Temescal/Telegraph     2313
2    Chinatown              1849
10   Jack London            1561
8    Fruitvale              1511
4    Koreatown/Northgate     850
6    Rockridge               603
9    Lakeshore               418
5    Montclair               389
11   Laurel                  303
Name: count, dtype: int64

## 6. Per-BID category distributions

One function, reused for every categorical field of interest: `broad_sector`, `employee_range`, `sales_range`, and `own_or_lease_da`.

For a given field, this:
1. Drops rows where that field is null (percentages are computed over non-null businesses only, not all businesses in the BID).
2. Counts businesses per `FID`/`BID` x category.
3. Converts counts to percentages of the (non-null) total within each `FID`/`BID`.
4. Pivots to one row per `FID`/`BID`, one column per category (named after the category's raw value).

In [32]:
def compute_bid_category_pct(gdf, category_col, id_cols=ID_COLS):
    """Return a wide dataframe of per-BID percentage makeup of `category_col`.

    Rows with a null value in `category_col` are dropped before computing.
    Output columns: id_cols..., <category1>, <category2>, ..., pct_sum_check
    (pct_sum_check is a QA column, dropped before saving to CSV; each row should
    sum to ~100). Category columns are named after the raw category values —
    no prefix.
    """
    id_cols = list(id_cols)

    if category_col not in gdf.columns:
        raise KeyError(
            f"'{category_col}' not found. Available columns: {list(gdf.columns)}"
        )

    subset = gdf.dropna(subset=[category_col])
    n_dropped = len(gdf) - len(subset)
    print(f"[{category_col}] dropped {n_dropped} rows with null values "
          f"({len(subset)} remaining)")

    counts = (
        subset
        .groupby(id_cols + [category_col])
        .size()
        .rename("count")
        .reset_index()
    )

    totals = counts.groupby(id_cols)["count"].sum().rename("total")
    counts = counts.merge(totals, on=id_cols)
    counts["pct"] = 100 * counts["count"] / counts["total"]

    wide = (
        counts
        .pivot(index=id_cols, columns=category_col, values="pct")
        .fillna(0)
    )
    wide.columns.name = None
    category_cols = list(wide.columns)
    wide = wide.reset_index()

    # QA check, not saved to the CSV
    wide["pct_sum_check"] = wide[category_cols].sum(axis=1)

    return wide

## 7. Run for every field and save one CSV each

Each output CSV keeps `FID`/`BID` as plain columns (no geometry), so it can be joined back onto the BID polygon file later using either key.

In [33]:
category_fields = ["broad_sector", "employee_range", "sales_range", "own_or_lease_da"]

category_results = {}

for field in category_fields:
    wide = compute_bid_category_pct(businesses_in_bids, field)
    category_results[field] = wide

    output_path = f"./static/business_statistics/{field}_distribution.csv"
    wide.drop(columns=["pct_sum_check"]).to_csv(output_path, index=False)

    print(f"Saved {len(wide)} rows to {output_path}")
    print("Columns:", list(wide.drop(columns=['pct_sum_check']).columns))
    print()

[broad_sector] dropped 0 rows with null values (15574 remaining)


OSError: Cannot save file into a non-existent directory: 'static/business_statistics'

In [ ]:
# Quick peek at each result
for field, wide in category_results.items():
    print(f"--- {field} ---")
    display(wide.head())

--- broad_sector ---


,FID,BID,Accommodation & Food Services,Administrative & Support Services,"Arts, Entertainment & Recreation",Construction,Education,"Finance, Insurance & Real Estate",Health Care & Social Assistance,Manufacturing,Other / Unclassified,Other Services,"Professional, Scientific & Technical Services",Public / Institutional,Retail Trade,Transportation & Warehousing,Wholesale Trade,pct_sum_check
0,1,Downtown,5.962311,3.552672,1.544640,1.915354,1.791783,16.558542,13.561940,1.482854,11.770158,9.051591,23.447637,2.749459,4.541242,0.864998,1.204819,100.0
1,2,Chinatown,5.624662,2.379665,1.027582,0.919416,1.460249,26.825311,17.577069,1.406165,10.383991,10.005408,7.517577,3.136831,9.356409,1.027582,1.352082,100.0
2,3,Lake Merritt,5.984252,2.913386,1.181102,1.929134,1.535433,15.944882,14.448819,1.574803,11.653543,11.496063,24.645669,0.511811,4.330709,0.748031,1.102362,100.0
3,4,Koreatown/Northgate,5.882353,1.882353,1.176471,0.823529,1.176471,21.882353,32.235294,1.058824,8.117647,9.529412,3.647059,0.352941,10.000000,0.823529,1.411765,100.0
4,5,Montclair,4.884319,2.827763,1.542416,3.084833,0.514139,21.850900,15.938303,2.056555,11.053985,12.596401,11.568123,0.257069,10.025707,0.514139,1.285347,100.0


--- employee_range ---


,FID,BID,1 to 4,10 to 19,100 to 249,1000 to 4999,20 to 49,250 to 499,5 to 9,50 to 99,500 to 999,pct_sum_check
0,1,Downtown,64.430894,7.977642,0.863821,0.050813,5.792683,0.355691,17.733740,2.693089,0.101626,100.0
1,2,Chinatown,72.261072,5.361305,0.699301,0.466200,4.428904,0.116550,14.918415,1.748252,0.000000,100.0
2,3,Lake Merritt,67.395498,6.945338,0.900322,0.192926,5.530547,0.128617,16.848875,1.993569,0.064309,100.0
3,4,Koreatown/Northgate,81.720430,4.301075,0.215054,0.000000,2.795699,0.000000,10.537634,0.430108,0.000000,100.0
4,5,Montclair,79.354839,3.225806,1.290323,0.000000,3.870968,0.000000,12.258065,0.000000,0.000000,100.0


--- sales_range ---


,FID,BID,$1-2.5 Million,$10-20 Million,$100-500 Million,$2.5-5 Million,$20-50 Million,$5-10 Million,$50-100 Million,"$500,000-1 Million",$500m - $1 Billion,"Less Than $500,000",pct_sum_check
0,1,Downtown,17.013233,1.764335,0.063012,5.923125,1.008192,2.268431,0.189036,20.856963,0.000000,50.913674,100.0
1,2,Chinatown,11.373708,0.886263,0.295421,4.283604,0.000000,1.920236,0.000000,13.737075,0.000000,67.503693,100.0
2,3,Lake Merritt,14.961832,1.603053,0.229008,5.725191,0.916031,3.206107,0.000000,21.068702,0.076336,52.213740,100.0
3,4,Koreatown/Northgate,8.373206,0.717703,0.000000,3.827751,0.000000,1.913876,0.000000,10.526316,0.000000,74.641148,100.0
4,5,Montclair,12.500000,1.470588,0.000000,1.470588,0.735294,2.941176,0.000000,16.176471,0.000000,64.705882,100.0


--- own_or_lease_da ---


,FID,BID,Lease,Own,Unknown,pct_sum_check
0,1,Downtown,41.497519,16.599008,41.903473,100.0
1,2,Chinatown,41.921859,16.156283,41.921859,100.0
2,3,Lake Merritt,39.206799,18.526912,42.266289,100.0
3,4,Koreatown/Northgate,21.906694,11.359026,66.734280,100.0
4,5,Montclair,25.301205,18.072289,56.626506,100.0
